# 01 — Dataset Exploration & Statistics

Exploratory analysis of the **GuitarDuets** dataset (Real + Synth subsets).
Visualizes waveforms, spectrograms, note annotations, and computes dataset-level statistics.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
import IPython.display as ipd

from src.data.manifests import load_manifest

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["figure.dpi"] = 100

## 1. Load the manifest

```bash
python scripts/build_metadata.py --config configs/dataset.yaml
```

In [ ]:
manifest = load_manifest(REPO_ROOT / "manifests" / "guitarrecordings.json", resolve_root=REPO_ROOT)
print(f"Total tracks: {len(manifest)}")

df = pd.DataFrame(manifest)
df["duration_s"] = df["length"] / df["samplerate"]
df["duration_min"] = df["duration_s"] / 60
df["has_notes"] = df["notes_csv"].notna()
df.head()

## 2. Dataset statistics

In [ ]:
for split in df["split"].unique():
    subset = df[df["split"] == split]
    total_min = subset["duration_min"].sum()
    notes_count = subset["has_notes"].sum()
    print(f"  {split}: {len(subset)} tracks, {total_min:.1f} min, "
          f"notes on {notes_count}/{len(subset)} tracks")

print(f"\nTotal duration: {df['duration_min'].sum():.1f} min")
print(f"Mean track duration: {df['duration_s'].mean():.1f}s ± {df['duration_s'].std():.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["duration_s"], bins=20, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Duration (seconds)")
axes[0].set_ylabel("Count")
axes[0].set_title("Track duration distribution")
axes[0].grid(True, alpha=0.3)

split_durations = df.groupby("split")["duration_min"].sum()
axes[1].bar(split_durations.index, split_durations.values, edgecolor="black", alpha=0.7)
axes[1].set_ylabel("Total duration (minutes)")
axes[1].set_title("Duration per split")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Visualize a track

In [ ]:
TRACK_IDX = 0
entry = manifest[TRACK_IDX]
print(f"Track: {entry['track_name']} (split={entry['split']})")

mix, sr = torchaudio.load(entry["mix"])
g1, _ = torchaudio.load(entry["sources"]["guitar1"])
g2, _ = torchaudio.load(entry["sources"]["guitar2"])

print(f"Sample rate: {sr} Hz, Duration: {mix.shape[-1]/sr:.2f}s")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
time = torch.arange(mix.shape[-1]) / sr

for ax, wav, label in zip(axes, [mix, g1, g2], ["Mix", "Guitar 1", "Guitar 2"]):
    ax.plot(time.numpy(), wav[0].numpy(), linewidth=0.3, alpha=0.8)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)")
axes[0].set_title(f"Waveforms — {entry['track_name']}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, wav, label in zip(axes, [mix, g1, g2], ["Mix", "Guitar 1", "Guitar 2"]):
    mono = wav.mean(dim=0)
    spec = torch.stft(mono, n_fft=2048, hop_length=512,
                      window=torch.hann_window(2048), return_complex=True)
    mag_db = 20 * torch.log10(spec.abs().clamp(min=1e-8))
    ax.imshow(mag_db.numpy(), aspect="auto", origin="lower",
              extent=[0, mix.shape[-1]/sr, 0, sr/2], cmap="magma", vmin=-60, vmax=0)
    ax.set_ylabel(f"{label}\nFreq (Hz)")
    ax.set_ylim(0, 8000)

axes[-1].set_xlabel("Time (s)")
axes[0].set_title(f"Spectrograms — {entry['track_name']}")
plt.tight_layout()
plt.show()

## 4. Listen

In [ ]:
print("Mix:")
ipd.display(ipd.Audio(mix.numpy(), rate=sr))
print("Guitar 1:")
ipd.display(ipd.Audio(g1.numpy(), rate=sr))
print("Guitar 2:")
ipd.display(ipd.Audio(g2.numpy(), rate=sr))

## 5. Note annotations (Synth tracks)

In [ ]:
synth_entries = [e for e in manifest if e.get("notes_csv")]
if synth_entries:
    entry_notes = synth_entries[0]
    notes_df = pd.read_csv(entry_notes["notes_csv"])
    print(f"Track: {entry_notes['track_name']}, Notes: {len(notes_df)}")
    display(notes_df.head(10))

    sr_notes = entry_notes["samplerate"]
    fig, ax = plt.subplots(figsize=(14, 5))
    for instr_id, color, label in [(1, "tab:blue", "Guitar 1"), (2, "tab:orange", "Guitar 2")]:
        subset = notes_df[notes_df["instrument"] == instr_id]
        for _, row in subset.iterrows():
            ax.barh(row["note"],
                    (row["end_time"] - row["start_time"]) / sr_notes,
                    left=row["start_time"] / sr_notes,
                    height=0.8, color=color, alpha=0.6)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("MIDI note")
    ax.set_title(f"Piano roll — {entry_notes['track_name']}")
    ax.legend([plt.Rectangle((0,0),1,1,fc="tab:blue",alpha=0.6),
               plt.Rectangle((0,0),1,1,fc="tab:orange",alpha=0.6)],
              ["Guitar 1", "Guitar 2"])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No tracks with note annotations found.")

## 6. Amplitude statistics

In [ ]:
stats = []
for entry in manifest[:10]:
    g1, _ = torchaudio.load(entry["sources"]["guitar1"])
    g2, _ = torchaudio.load(entry["sources"]["guitar2"])
    stats.append({
        "track": entry["track_name"],
        "g1_rms": float(g1.pow(2).mean().sqrt()),
        "g2_rms": float(g2.pow(2).mean().sqrt()),
        "g1_peak": float(g1.abs().max()),
        "g2_peak": float(g2.abs().max()),
    })

stats_df = pd.DataFrame(stats)
stats_df["rms_ratio"] = stats_df["g1_rms"] / stats_df["g2_rms"].clip(lower=1e-8)
display(stats_df)
print(f"\nMean G1/G2 RMS ratio: {stats_df['rms_ratio'].mean():.3f} ± {stats_df['rms_ratio'].std():.3f}")